In [1]:
import re
import numpy as np
import os
import torch
import tempfile
from pathlib import Path
from datasets import load_dataset
from moviepy.video.io.VideoFileClip import VideoFileClip
from IPython.display import Video, display
from safetensors.torch import load_file

from src.config import Config
from src.model.virality_predictor import ViralityPredictor
from src.model.data_processor import DataProcessor

# 1. Setup Device
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

# 2. Load Config and Model
config = Config()
checkpoint_dir = Path(config.checkpoint_path)
checkpoint_folders = [f for f in checkpoint_dir.iterdir()
                      if f.is_dir() and re.match(r'checkpoint-\d+$', f.name)]
latest_checkpoint = max(checkpoint_folders, key=lambda f: int(
    f.name.split('-')[1])) if checkpoint_folders else None
assert latest_checkpoint, f'No checkpoint found inside {checkpoint_dir}'

print(f'Loading model from: {latest_checkpoint}')
model = ViralityPredictor(config)
state_dict = load_file(latest_checkpoint / 'model.safetensors')
model.load_state_dict(state_dict)
model.to(device)
model.eval()

# 3. Load Data & Processor
dataset = load_dataset(config.dataset_id)['train']
processor = DataProcessor(config)

# Choose a random example
example_idx = np.random.randint(0, len(dataset))
example = dataset[example_idx]

print(f'\n=== Ground Truth (Example {example_idx}) ===')
print(f'Engagement: {example["engagement_score"]:.4f}')
print(f'Velocity: {example["view_velocity_score"]:.4f}')
print(f'Is Viral: {bool(example["is_viral"])}')

# 4. Prepare Input Batch
# We wrap single values in lists to mimic a batch
batch = {k: [v] for k, v in example.items()}

# Use the internal processing logic (ensuring we match the trainer's format)
processed = processor._process_batch(batch)
inference_batch = {
    k: v.to(device) for k, v in processed.items() if k != 'labels'
}

# 5. Run Inference
with torch.no_grad():
    output = model(**inference_batch, labels=None)

    # Extract from new multi-head structure
    # [batch, 2] -> [engagement, velocity]
    regression_logits = output['regression_logits']
    # [batch, 1] -> [viral_logit]
    classification_logits = output['classification_logits']

# 6. Post-Process Predictions
# Invert the log1p transformation applied during data processing
pred_engagement, pred_velocity = np.expm1(regression_logits[0].cpu().numpy())

# Convert raw logit to probability [0, 1]
viral_prob = torch.sigmoid(classification_logits[0]).item()

# ADJUST THIS: Higher threshold = Higher Precision, Lower Recall
PRECISION_THRESHOLD = 0.7
predicted_viral = viral_prob >= PRECISION_THRESHOLD

print(f'\n=== Model Predictions ===')
print(f'Pred Engagement: {pred_engagement:.4f}')
print(f'Pred Velocity:   {pred_velocity:.4f}')
print(f'Viral Confidence: {viral_prob:.2%}')
print(f'Classification:  {"VIRAL" if predicted_viral else "NOT VIRAL"}')

# 7. Validation Logic
ground_truth_viral = bool(example['is_viral'])
match = (ground_truth_viral == predicted_viral)
print(f'\n=== Evaluation ===')
print(f'Prediction Match: {match}')
if not match and predicted_viral:
    print("Result: False Positive (Low Precision)")
elif not match and not predicted_viral:
    print("Result: False Negative (Low Recall)")
else:
    print("Result: Correct Classification")

# 8. Video Preview
if example['video_bytes']:
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as tmp:
        tmp.write(example['video_bytes'])
        tmp_path = tmp.name
    try:
        display(Video(tmp_path, embed=True, width=300))
    finally:
        os.unlink(tmp_path)

Using device: mps
Loading model from: data/checkpoints/checkpoint-7806


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/184 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.q_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.v_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.bias      | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.weight        | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.query.weight | UNEXPECTED |  | 
decoder.norm.bias                                                    | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.weight       

RuntimeError: Error(s) in loading state_dict for ViralityPredictor:
	Missing key(s) in state_dict: "pos_weight", "late_fusion_mlp.0.weight", "late_fusion_mlp.0.bias", "engagement_head.0.weight", "engagement_head.0.bias", "engagement_head.2.weight", "engagement_head.2.bias", "velocity_head.0.weight", "velocity_head.0.bias", "velocity_head.2.weight", "velocity_head.2.bias", "classification_head.0.weight", "classification_head.0.bias", "classification_head.2.weight", "classification_head.2.bias", "classification_loss.pos_weight". 
	Unexpected key(s) in state_dict: "classifier.0.bias", "classifier.0.weight", "classifier.3.bias", "classifier.3.weight". 
	size mismatch for tabular_mlp.0.weight: copying a param with shape torch.Size([64, 17]) from checkpoint, the shape in current model is torch.Size([512, 19]).
	size mismatch for tabular_mlp.0.bias: copying a param with shape torch.Size([64]) from checkpoint, the shape in current model is torch.Size([512]).
	size mismatch for tabular_mlp.2.we